In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

In [11]:
import csv
from collections import defaultdict

ALL_MAPPED = 'all_mapped.txt'

# form_id -> set of global writer IDs mentioned inside that form
form_to_global = defaultdict(set)

with open(ALL_MAPPED, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    for global_writer, cell_id, *_ in reader:
        form_id = cell_id.split('_', 1)[0]
        form_to_global[form_id].add(global_writer)

len(form_to_global), next(iter(form_to_global.items()))


(14,
 ('0',
  {'0000',
   '0001',
   '00010',
   '000100',
   '000101',
   '000102',
   '000103',
   '00011',
   '00012',
   '00013',
   '00014',
   '00015',
   '00016',
   '00017',
   '00018',
   '00019',
   '0002',
   '00020',
   '00021',
   '00022',
   '00023',
   '00024',
   '00025',
   '00026',
   '00027',
   '00028',
   '00029',
   '0003',
   '00030',
   '00031',
   '00032',
   '00033',
   '00034',
   '00035',
   '00036',
   '00037',
   '00038',
   '00039',
   '0004',
   '00040',
   '00041',
   '00042',
   '00043',
   '00044',
   '00045',
   '00046',
   '00047',
   '00048',
   '00049',
   '0005',
   '00050',
   '00051',
   '00052',
   '00053',
   '00054',
   '00055',
   '00056',
   '00057',
   '00058',
   '00059',
   '0006',
   '00060',
   '00061',
   '000611',
   '000612',
   '000613',
   '000614',
   '000615',
   '000616',
   '000617',
   '000618',
   '000619',
   '00062',
   '000620',
   '000621',
   '000622',
   '000623',
   '000624',
   '000625',
   '000626',
   '000627',
  

In [12]:
import networkx as nx

G = nx.Graph()
for form_id, writers in form_to_global.items():
    G.add_node(form_id, writers=writers)

# Add edges between forms that share any global writer id
global_to_forms = defaultdict(list)
for form_id, writers in form_to_global.items():
    for gw in writers:
        global_to_forms[gw].append(form_id)

for forms in global_to_forms.values():
    if len(forms) > 1:
        for i in range(len(forms)):
            for j in range(i + 1, len(forms)):
                G.add_edge(forms[i], forms[j])

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
num_components = nx.number_connected_components(G)
num_nodes, num_edges, num_components


(14, 0, 14)

In [ ]:
form_cluster = {}        # form_id -> cluster_id
cluster_members = []     # list of sets of form ids

for cluster_id, nodes in enumerate(nx.connected_components(G)):
    cluster_members.append(nodes)
    for form_id in nodes:
        form_cluster[form_id] = cluster_id

len(cluster_members), cluster_members[:5]


(14, [{'0'}, {'10'}, {'1'}, {'11'}, {'12'}])

In [14]:
import json
import os

REMAPPING_JSON = 'writer_cluster_map.json'

sample_to_cluster = {}
with open(ALL_MAPPED, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    for global_writer, cell_id, *_ in reader:
        form_id = cell_id.split('_', 1)[0]
        cluster_id = form_cluster[form_id]
        filename = f'{cell_id}.jpg'
        sample_to_cluster[filename] = cluster_id

with open(REMAPPING_JSON, 'w', encoding='utf-8') as out:
    json.dump({'form_cluster': form_cluster,
               'sample_cluster': sample_to_cluster}, out, ensure_ascii=False, indent=2)

REMAPPING_JSON


'writer_cluster_map.json'

In [16]:
%pip install pandas

  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.4 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd

# cluster size distribution
sizes = [len(nodes) for nodes in cluster_members]
pd.Series(sizes).describe(), pd.Series(sizes).value_counts().head()

# how many consolidated writer ids?
num_clusters = len(cluster_members)
num_clusters


14

In [3]:
from style_encoder_train_cyrillic import HKRDataset_style, ImageEncoder

BATCH = 512
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = HKRDataset_style(
    basefolder='hkr-dataset',
    subset='train',
    segmentation_level='word',
    fixed_size=(64, 256),
    transforms=transforms.Compose([transforms.ToTensor()])
)

loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, num_workers=4)

ckpt = 'style_models/triplet_hkr_mobilenetv2_100.pth'
encoder = ImageEncoder('mobilenetv2_100', num_classes=0, pretrained=False, trainable=False).to(DEVICE)
encoder.load_state_dict(torch.load(ckpt, map_location=DEVICE))
encoder.eval()

/home/oles/DiffusionPen/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


num_tokens 1
save_file ./IAM_dataset_PIL_style/train_word_HKR_v2.pt
Number of writers 1614


ImageEncoder(
  (model): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU6(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): ReLU6(inplace=True)
          )
          (aa): Identity()
          (se): Identity()
          (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn2): BatchNormAct2d(
            16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): Identity()
   

In [4]:
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path

form_embeddings = defaultdict(list)

with torch.no_grad():
    for images, _, _, _, _, _, _, _, paths, *_ in tqdm(loader):
        images = images.to(DEVICE)
        feats = encoder(images).view(images.size(0), -1).cpu().numpy()

        for feat, path in zip(feats, paths):
            filename = Path(path).name  # e.g. 0_12_0.jpg
            form_id = filename.split('_', 1)[0]
            form_embeddings[form_id].append(feat)

# average all word-level embeddings inside each form
form_vectors = {form_id: np.mean(vecs, axis=0) for form_id, vecs in form_embeddings.items()}
len(form_vectors)


100%|██████████| 127/127 [04:55<00:00,  2.33s/it]


14

In [5]:
np.savez('form_embeddings.npz', **form_vectors)

In [7]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 10.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 9.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [9]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_distances

form_ids = sorted(form_vectors.keys())
X = np.stack([form_vectors[fid] for fid in form_ids])

clustering = AgglomerativeClustering(
    n_clusters=None,
    metric='cosine',
    linkage='average',
    distance_threshold=0.2,
    compute_distances=True
)
cluster_labels = clustering.fit_predict(X)


len(set(cluster_labels))


1

In [18]:
form_cluster = {fid: int(label) for fid, label in zip(form_ids, cluster_labels)}

sample_cluster = {}
for filename, cluster_id in sample_to_cluster.items():
    form_id = filename.split('_', 1)[0]
    sample_cluster[filename] = form_cluster[form_id]

with open('writer_cluster_map_embedded.json', 'w', encoding='utf-8') as out:
    json.dump({'form_cluster': form_cluster,
               'sample_cluster': sample_cluster}, out, ensure_ascii=False, indent=2)

print('Clusters:', len(set(cluster_labels)))


Clusters: 1


In [19]:
import random
def show_cluster_examples(target_cluster, k=4):
    members = [idx for idx, cid in enumerate(cluster_labels) if cid == target_cluster]
    display_members = random.sample(members, min(len(members), k))
    print(f'Cluster {target_cluster}, forms: {[form_ids[i] for i in display_members]}')

show_cluster_examples(target_cluster=0)


Cluster 0, forms: ['2', '0', '5', '10']


In [20]:
norms = {fid: np.linalg.norm(vec) for fid, vec in form_vectors.items()}
pd.Series(norms).describe()


count    14.000000
mean     57.081806
std       0.164096
min      56.877426
25%      56.973366
50%      57.033575
75%      57.155365
max      57.422424
dtype: float64

In [21]:
norms

{'0': np.float32(57.185024),
 '10': np.float32(56.973297),
 '11': np.float32(56.877426),
 '12': np.float32(56.922935),
 '13': np.float32(56.97357),
 '1': np.float32(57.006634),
 '2': np.float32(57.29077),
 '3': np.float32(57.422424),
 '4': np.float32(57.33533),
 '5': np.float32(57.046818),
 '6': np.float32(57.058758),
 '7': np.float32(57.066387),
 '8': np.float32(57.020332),
 '9': np.float32(56.96554)}

In [22]:
from sklearn.metrics.pairwise import cosine_distances
D = cosine_distances(X)  # 14 x 14
triu = D[np.triu_indices_from(D, k=1)]
print(triu.min(), triu.max(), triu.mean())


9.357929e-06 0.0006380081 0.00023665205


In [24]:
for thr in [6e-4, 4e-4, 3e-4, 2e-4, 1e-4, 5e-5]:
    labels = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=thr
    ).fit_predict(X)  # or X if you keep raw vectors
    print(f'{thr:.1e} -> clusters: {len(set(labels))}')


6.0e-04 -> clusters: 1
4.0e-04 -> clusters: 2
3.0e-04 -> clusters: 3
2.0e-04 -> clusters: 3
1.0e-04 -> clusters: 5
5.0e-05 -> clusters: 8


Let's run for all now:

In [34]:
import torch, numpy as np, json
from pathlib import Path
from collections import defaultdict
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import csv

from style_encoder_train_cyrillic import HKRDataset_style, ImageEncoder

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH = 512

dataset = HKRDataset_style('hkr-dataset', 'train', 'word', (64, 256),
                           transforms=transforms.Compose([transforms.ToTensor()]))

loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, num_workers=4)

encoder = ImageEncoder('mobilenetv2_100', num_classes=0,
                       pretrained=False, trainable=False).to(DEVICE)
encoder.load_state_dict(torch.load('style_models/triplet_hkr_mobilenetv2_100.pth',
                                   map_location=DEVICE))
encoder.eval()

filename_to_writer = {}
with open('all_mapped.txt', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    for global_writer, cell_id, *_ in reader:
        filename = f'{cell_id}.jpg'
        filename_to_writer[filename] = global_writer


writer_embeddings = defaultdict(list)

with torch.no_grad():
    for images, _, _, _, _, _, _, _, paths, *_ in tqdm(loader):
        feats = encoder(images.to(DEVICE)).view(images.size(0), -1).cpu().numpy()
        for feat, path in zip(feats, paths):
            filename = Path(path).name
            writer_id = filename_to_writer.get(filename)
            if writer_id is None:
                continue
            writer_embeddings[writer_id].append(feat)

writer_vectors = {wid: np.mean(vecs, axis=0) for wid, vecs in writer_embeddings.items()}
np.savez('writer_vectors.npz', **writer_vectors)
len(writer_vectors) # should be ~1.6k


save_file ./IAM_dataset_PIL_style/train_word_HKR_v2.pt
Number of writers 1614


100%|██████████| 127/127 [07:12<00:00,  3.41s/it]


1614

In [37]:
data = np.load('writer_vectors.npz')
form_vectors = {fid: data[fid] for fid in data.files}

In [38]:
len(form_vectors)

1614

In [40]:
len(dataset), len(form_embeddings), sorted(form_embeddings.keys())


(64943,
 14,
 ['0', '1', '10', '11', '12', '13', '2', '3', '4', '5', '6', '7', '8', '9'])

Going to sweep across really tiny thresolds as it showed to be the right direction earlier:

In [45]:
from sklearn.preprocessing import normalize

form_ids = sorted(form_vectors.keys())
X = np.stack([form_vectors[fid] for fid in form_ids])
X_norm = normalize(X)

def cluster_with_threshold(threshold):
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=threshold,
        compute_distances=False
    )
    labels = clustering.fit_predict(X_norm)
    return labels

for thr in [1e-4, 2e-4, 4e-4, 1e-3, 2e-3, 5e-3]:
    labels = cluster_with_threshold(thr)
    print(f'{thr:.1e} -> clusters: {len(set(labels))}')


1.0e-04 -> clusters: 992
2.0e-04 -> clusters: 589
4.0e-04 -> clusters: 268
1.0e-03 -> clusters: 72
2.0e-03 -> clusters: 30
5.0e-03 -> clusters: 7


In [46]:
chosen_thr = 4e-4   # or 2e-4
cluster_labels = cluster_with_threshold(chosen_thr)
print('Final clusters:', len(set(cluster_labels)))

Final clusters: 268


In [48]:
form_cluster = {writer_id: int(label) for writer_id, label in zip(form_ids, cluster_labels)}
sample_cluster = {}

with open('all_mapped.txt', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    for global_writer, cell_id, *_ in reader:
        filename = f'{cell_id}.jpg'
        cluster_id = form_cluster.get(global_writer)
        if cluster_id is None:
            continue  # skip if this writer wasn't in the embedding set
        sample_cluster[filename] = cluster_id


with open('writer_cluster_map_embedded.json', 'w', encoding='utf-8') as out:
    json.dump({
        'params': {'threshold': chosen_thr, 'num_clusters': len(set(cluster_labels))},
        'form_cluster': form_cluster,
        'sample_cluster': sample_cluster
    }, out, ensure_ascii=False, indent=2)
